In [2]:
pip install wbgapi

In [3]:
import pandas as pd

In [4]:
import wbgapi as wb

In [18]:
indicators =['NY.GDP.PCAP.CD','SI.POV.DDAY']

In [19]:
df= wb.data.DataFrame(indicators,time=range(2010,2025),labels=True)

In [20]:
df=df.reset_index()

In [21]:
df=df.rename(columns={
    'NY.GDP.PCAP.CD':'GDP',
    'SI,POV,DDAY':'Poverty',
    'Country':'Country_name'
})

In [22]:

df.to_csv('world_bank_macro_data.csv',index=False)

In [23]:
df.head()

,economy,series,Country_name,Series,YR2010,YR2011,YR2012,YR2013,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
0,ZWE,NY.GDP.PCAP.CD,Zimbabwe,GDP per capita (current US$),902.010759,1037.775652,1239.227097,1362.994983,1372.915262,1387.126326,1408.139453,3445.449410,2270.895319,2184.521554,2059.637040,2613.616741,2536.401971,2195.225020,2496.155150
1,ZMB,NY.GDP.PCAP.CD,Zambia,GDP per capita (current US$),1451.106160,1624.868178,1710.050613,1820.718548,1707.485731,1295.877887,1239.085279,1483.465773,1463.899979,1258.986198,951.644317,1127.160779,1447.123101,1330.727806,1187.109434
2,YEM,NY.GDP.PCAP.CD,"Yemen, Rep.",GDP per capita (current US$),1155.203053,1186.475006,1245.050683,1378.750030,1430.164210,1362.173812,975.359417,811.165970,633.887202,NaN,NaN,NaN,NaN,NaN,NaN
3,PSE,NY.GDP.PCAP.CD,West Bank and Gaza,GDP per capita (current US$),2557.075624,2880.798437,3067.438727,3315.297539,3352.112595,3272.154324,3527.613824,3620.360487,3562.330943,3656.858271,3233.568638,3678.635657,3799.955270,3607.416119,3028.254813
4,VIR,NY.GDP.PCAP.CD,Virgin Islands (U.S.),GDP per capita (current US$),39905.128418,38997.137316,37795.319259,34597.976694,33045.364380,34007.352941,35324.974887,35365.069304,36663.208755,38633.529892,39787.374165,42571.077737,44320.909186,NaN,NaN


In [24]:
df.columns

Index(['economy', 'series', 'Country_name', 'Series', 'YR2010', 'YR2011',
       'YR2012', 'YR2013', 'YR2014', 'YR2015', 'YR2016', 'YR2017', 'YR2018',
       'YR2019', 'YR2020', 'YR2021', 'YR2022', 'YR2023', 'YR2024'],
      dtype='object')

In [25]:
year_columns = [f'YR{year}' for year in range(2010, 2025)]
df_long = pd.melt(
    df,
    id_vars=['economy', 'Country_name', 'series'],
    value_vars=year_columns,
    var_name='Year',
    value_name='Value'
)

In [26]:
df_long['Year'] = df_long['Year'].str.replace('YR', '').astype(int)

In [27]:
df_final = df_long.pivot_table(
    index=['economy', 'Country_name', 'Year'],
    columns='series',
    values='Value'
).reset_index()

In [28]:
df_final = df_final.rename(columns={
    'NY.GDP.PCAP.CD': 'GDP_Per_Capita',
    'SI.POV.DDAY': 'Poverty_Headcount_Ratio',
    'economy': 'Country_ISO'
})

In [29]:
df_final.to_csv("cleaned_world_bank_data.csv", index=False)

In [30]:
df_final.head()

series,Country_ISO,Country_name,Year,GDP_Per_Capita,Poverty_Headcount_Ratio
0,ABW,Aruba,2010,24093.140151,NaN
1,ABW,Aruba,2011,25712.384302,NaN
2,ABW,Aruba,2012,25119.665545,NaN
3,ABW,Aruba,2013,25813.571441,NaN
4,ABW,Aruba,2014,26129.839062,NaN


In [31]:
import numpy as np


In [32]:
df_clean = pd.read_csv("cleaned_world_bank_data.csv")

In [33]:
df_clean = df_clean.sort_values(by=['Country_ISO', 'Year']).reset_index(drop=True)

In [34]:
df_clean['GDP_Per_Capita'] = df_clean.groupby('Country_ISO')['GDP_Per_Capita'].transform(lambda x: x.interpolate(method='linear').ffill().bfill())
df_clean['Poverty_Headcount_Ratio'] = df_clean.groupby('Country_ISO')['Poverty_Headcount_Ratio'].transform(lambda x: x.interpolate(method='linear').ffill().bfill())

In [35]:
import pandas as pd


In [39]:
try:
    df_who = pd.read_excel("GHED all data (March 2026).xlsx")
except Exception as e:
    print(e)

In [40]:
df_who = pd.read_excel("GHED all data (March 2026).xlsx")

In [41]:
df_who.columns = df_who.columns.str.strip()

In [42]:
df_who_filtered = df_who[['code', 'year', 'oops_che']].copy()

In [43]:
df_who_filtered.rename(columns={
    'code': 'Country_ISO',
    'year': 'Year',
    'oops_che': 'OOP_Percentage'
}, inplace=True)

In [44]:
df_who_filtered['Year'] = df_who_filtered['Year'].astype(int)
df_who_filtered['Country_ISO'] = df_who_filtered['Country_ISO'].astype(str).str.upper()
df_who_filtered['OOP_Percentage'] = pd.to_numeric(df_who_filtered['OOP_Percentage'], errors='coerce')

In [46]:
df_wb = pd.read_csv("cleaned_world_bank_data.csv")
df_wb['Country_ISO'] = df_wb['Country_ISO'].astype(str).str.upper()
df_wb['Year'] = df_wb['Year'].astype(int)

In [47]:
df_master = pd.merge(df_wb, df_who_filtered, on=['Country_ISO', 'Year'], how='inner')

In [48]:
df_master.dropna(subset=['OOP_Percentage'], inplace=True)

In [49]:
df_master.to_csv("master_health_economics_dataset.csv", index=False)

In [50]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

In [51]:
df_master = pd.read_csv("master_health_economics_dataset.csv")

In [52]:
features = ['GDP_Per_Capita', 'Poverty_Headcount_Ratio', 'Year']
X = df_master[features]
y = df_master['OOP_Percentage']

In [53]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [54]:
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples\n")

Training set size: 2137 samples
Testing set size: 535 samples



Model 1= Random forest regressor


In [55]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

Moderl 2= Xgboost regressor

In [56]:
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.08, max_depth=5, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

In [57]:
def calculate_metrics(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {
        "Model": model_name,
        "MAE (% OOP)": round(mae, 3),
        "RMSE (% OOP)": round(rmse, 3),
        "R² Score": round(r2, 3)
    }

In [58]:
results = [
    calculate_metrics(y_test, rf_preds, "Random Forest"),
    calculate_metrics(y_test, xgb_preds, "XGBoost Regressor")
]


In [59]:
df_results = pd.DataFrame(results)
print(" MACHINE LEARNING MODEL PERFORMANCE ")
print(df_results.to_string(index=False))

 MACHINE LEARNING MODEL PERFORMANCE 
            Model  MAE (% OOP)  RMSE (% OOP)  R² Score
    Random Forest       12.399        16.878     0.158
XGBoost Regressor       12.092        16.218     0.223


In [61]:
importance = xgb_model.feature_importances_
print("\n XGBOOST FEATURE IMPORTANCE ")
for feat, imp in zip(features, importance):
    print(f"{feat}: {round(imp * 100, 2)}%")


 XGBOOST FEATURE IMPORTANCE 
GDP_Per_Capita: 61.650001525878906%
Poverty_Headcount_Ratio: 25.850000381469727%
Year: 12.510000228881836%
